# Vision-Guided Pick-and-Place System
This notebook implements the full vision-guided pick-and-place system for the OpenManipulator-X, configured as requested in `template.txt`.
It integrates RealSense vision, ArUco detection, and Dynamixel SDK control.

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import math
import time
from dynamixel_sdk import *

In [2]:
# ============================================================
# 1. REALSENSE SETUP
# ============================================================
def init_realsense():
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

    profile = pipeline.start(config)
    align_to = rs.stream.color
    align = rs.align(align_to)

    intrinsics = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()
    return pipeline, align, intrinsics

def get_frames(pipeline, align):
    frames = pipeline.wait_for_frames()
    aligned_frames = align.process(frames)
    color_frame = aligned_frames.get_color_frame()
    depth_frame = aligned_frames.get_depth_frame()

    if not color_frame or not depth_frame:
        return None, None
    color_image = np.asanyarray(color_frame.get_data())
    return color_image, depth_frame

In [3]:
# ============================================================
# 2. ARUCO DETECTION
# ============================================================
def init_aruco():
    try:
        aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(aruco_dict, detector_params)
        return detector_params, aruco_dict, detector, False
    except AttributeError:
        aruco_dict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters_create()
        return detector_params, aruco_dict, None, True

def detect_markers(image, aruco_dict, detector, OPENCV_OLD, detector_params):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if OPENCV_OLD:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=detector_params)
    else:
        corners, ids, rejected = detector.detectMarkers(gray)
    return corners, ids

In [4]:
# ============================================================
# 3. WORKSPACE MAPPING (HOMOGRAPHY)
# ============================================================
def compute_homography(corners, ids, W=0.3, H=0.2):
    expected_ids = [0, 1, 2, 3]
    if ids is None: return None
    
    detected_ids = ids.flatten().tolist()
    if not all(req_id in detected_ids for req_id in expected_ids): return None

    image_pts = []
    world_pts = np.array([
        [0, 0], [W, 0], [W, H], [0, H]
    ], dtype=np.float32)

    for req_id in expected_ids:
        idx = detected_ids.index(req_id)
        marker_corners = corners[idx][0]
        center_x = np.mean(marker_corners[:, 0])
        center_y = np.mean(marker_corners[:, 1])
        image_pts.append([center_x, center_y])

    image_pts = np.array(image_pts, dtype=np.float32)
    H_matrix, _ = cv2.findHomography(image_pts, world_pts)
    return H_matrix

def pixel_to_workspace(u, v, H_matrix):
    pt = np.array([[[u, v]]], dtype=np.float32)
    pt_trans = cv2.perspectiveTransform(pt, H_matrix)
    return pt_trans[0][0][0], pt_trans[0][0][1]

In [5]:
# ============================================================
# 4. DEPTH & 5. ROBOT BASE CALIBRATION & 6. OBJECT POSITION
# ============================================================
def get_depth(depth_frame, u, v):
    return depth_frame.get_distance(u, v)

def get_base_transform(corners, ids, marker_size=0.05, camera_matrix=None, dist_coeffs=None):
    if ids is None: return None
    detected_ids = ids.flatten().tolist()
    if 4 not in detected_ids: return None

    idx = detected_ids.index(4)
    # Define the 3d coordinates of the 4 corners of the marker
    marker_points = np.array([
        [-marker_size / 2, marker_size / 2, 0],
        [marker_size / 2, marker_size / 2, 0],
        [marker_size / 2, -marker_size / 2, 0],
        [-marker_size / 2, -marker_size / 2, 0]
    ], dtype=np.float32)
    
    # Use solvePnP to get pose (replacing deprecated cv2.aruco.estimatePoseSingleMarkers)
    success, rvec, tvec = cv2.solvePnP(marker_points, corners[idx][0], camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_IPPE_SQUARE)
    if not success: return None
    
    rvec = rvec.flatten()
    tvec = tvec.flatten()
    
    R, _ = cv2.Rodrigues(rvec)
    T_camera_base = np.eye(4)
    T_camera_base[:3, :3] = R
    T_camera_base[:3, 3] = tvec
    
    # Invert
    return np.linalg.inv(T_camera_base)

def pixel_to_camera(u, v, Z, intrinsics):
    pt = rs.rs2_deproject_pixel_to_point(intrinsics, [u, v], Z)
    return np.array([pt[0], pt[1], pt[2], 1.0])

def transform_to_base(pt_camera, T_base_camera):
    P_base = T_base_camera @ pt_camera
    return P_base[:3]

In [6]:
# ============================================================
# 8. INVERSE KINEMATICS (OpenManipulator-X)
# ============================================================
def inverse_kinematics(x, y, z):
    L1, L2, L3 = 0.077, 0.130, 0.124
    base_height = L1
    
    joint1 = math.atan2(y, x)
    r = math.sqrt(x**2 + y**2)
    z_prime = z - base_height
    
    D = (r**2 + z_prime**2 - L2**2 - L3**2) / (2 * L2 * L3)
    D = max(-1.0, min(1.0, D))
    
    # Inverted elbow solution (chooses the opposite kinematics pose)
    joint3 = math.atan2(-math.sqrt(1 - D**2), D)
    joint2 = math.atan2(z_prime, r) - math.atan2(L3 * math.sin(joint3), L2 + L3 * math.cos(joint3))
    
    joint4 = -(joint2 + joint3)
    return [joint1, joint2, joint3, joint4]

In [7]:
# ============================================================
# 9. DYNAMIXEL SDK SETUP & 10. POSITION & 11. SMOOTH MOTION
# ============================================================
ADDR_TORQUE_ENABLE = 64
ADDR_GOAL_POSITION = 116
ADDR_PRESENT_POSITION = 132
PROTOCOL_VERSION = 2.0
BAUDRATE = 1000000
DEVICENAME = 'COM9' # Match robot.ipynb
DXL_IDs = [11, 12, 13, 14, 15]

def init_dynamixel():
    portHandler = PortHandler(DEVICENAME)
    packetHandler = PacketHandler(PROTOCOL_VERSION)
    
    if not portHandler.openPort() or not portHandler.setBaudRate(BAUDRATE):
        return None, None
        
    for dxl_id in DXL_IDs:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, 1)
    return portHandler, packetHandler

def rad_to_dynamixel(angle):
    pos = int((angle + math.pi) * (4095.0 / (2 * math.pi)))
    return max(0, min(4095, pos))

def smooth_profile(t):
    return 0.5 * (1 - math.cos(math.pi * t))

def send_joint_positions(portHandler, packetHandler, q_positions):
    for i, dxl_id in enumerate([11, 12, 13, 14]):
        pos = rad_to_dynamixel(q_positions[i])
        packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, pos)

def move_smooth(portHandler, packetHandler, q_start, q_goal, duration):
    steps = 80
    dt = duration / steps
    for i in range(steps):
        t = i / steps
        alpha = smooth_profile(t)
        q = [qs + alpha * (qg - qs) for qs, qg in zip(q_start, q_goal)]
        send_joint_positions(portHandler, packetHandler, q)
        time.sleep(dt)

def set_gripper(portHandler, packetHandler, open_gripper=True):
    pos = 1228 if open_gripper else 2500
    packetHandler.write4ByteTxRx(portHandler, 15, ADDR_GOAL_POSITION, pos)

def read_positions(portHandler, packetHandler):
    positions = {}
    for dxl_id in DXL_IDs:
        pos, dxl_comm, dxl_err = packetHandler.read4ByteTxRx(portHandler, dxl_id, ADDR_PRESENT_POSITION)
        if dxl_comm == COMM_SUCCESS and dxl_err == 0:
            positions[dxl_id] = pos
    return positions

def set_default_positions(portHandler, packetHandler):
    default_pos = {11: 2048, 12: 1227, 13: 2524, 14: 2414, 15: 1228}
    print("Moving robot to default position using software interpolation...")
    
    current_positions = read_positions(portHandler, packetHandler)
    if not current_positions:
        print("Failed to read current positions. Moving directly to default...")
        for dxl_id, pos in default_pos.items():
            packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, pos)
        time.sleep(1.5)
        return

    steps = 50
    step_delay = 0.03
    increments = {}
    
    for dxl_id in default_pos:
        if dxl_id in current_positions:
            increments[dxl_id] = (default_pos[dxl_id] - current_positions[dxl_id]) / steps
            
    for step in range(1, steps + 1):
        for dxl_id in default_pos:
            if dxl_id in increments:
                int_pos = int(current_positions[dxl_id] + (increments[dxl_id] * step))
                int_pos = max(0, min(4095, int_pos))
                packetHandler.write4ByteTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, int_pos)
        time.sleep(step_delay)

In [8]:
# ============================================================
# 14. MOTION SEQUENCE
# ============================================================
def dynamixel_to_rad(pos):
    return (pos * (2 * math.pi) / 4095.0) - math.pi

def execute_task(pipeline, align, intrinsics, detector_params, aruco_dict, detector, OPENCV_OLD, portHandler, packetHandler, camera_matrix, dist_coeffs):
    """Executes the CV and target movement tasks. Returns boolean on success."""
    color_image, depth_frame = get_frames(pipeline, align)
    if color_image is None:
        return False
        
    corners, ids = detect_markers(color_image, aruco_dict, detector, OPENCV_OLD, detector_params)
    
    # --- VISUALIZATION ---
    display_image = color_image.copy()
    if ids is not None:
        cv2.aruco.drawDetectedMarkers(display_image, corners, ids)
    cv2.imshow("Camera Feed - ArUco Detection", display_image)
    cv2.waitKey(1)
    # ---------------------
    
    H_matrix = compute_homography(corners, ids)
    T_base_cam = get_base_transform(corners, ids, 0.05, camera_matrix, dist_coeffs)
    
    if H_matrix is None or T_base_cam is None:
        # Keep searching quietly until found
        return False
        
    # --- FIND TARGET ARUCO MARKER (Any ID > 4) ---
    target_id = None
    detected_ids = ids.flatten().tolist()
    
    for i, marker_id in enumerate(detected_ids):
        if marker_id not in [0, 1, 2, 3, 4]:
            target_id = marker_id
            marker_corners = corners[i][0]
            u = int(np.mean(marker_corners[:, 0]))
            v = int(np.mean(marker_corners[:, 1]))
            break
            
    if target_id is None:
        # Keep searching until a target object is placed in view
        return False
        
    print(f"Target object detected (ID: {target_id}) at pixel ({u}, {v})")
    Z = get_depth(depth_frame, u, v)
    
    if Z <= 0:
        return False
        
    pt_cam = pixel_to_camera(u, v, Z, intrinsics)
    P_base = transform_to_base(pt_cam, T_base_cam)
    tx, ty, tz = P_base[0], P_base[1], P_base[2] + 0.05
    
    # Fix the 90-degree rotated ArUco Marker mapping!
    # Swap X and Y and negate the new X to correct the coordinate system to match physical robot movement
    tx, ty = -ty, tx 
    
    # Print the coordinates for debugging
    print(f"Base Detected! Corrected Target -> X: {tx:.3f}m, Y: {ty:.3f}m, Z: {tz:.3f}m")
    
    # SAFETY FALLBACK: Do not initiate pick up if the target is in negative coordinates
    if tx < 0 or ty < 0:
        print("Target is in negative coordinates (X < 0 or Y < 0). Aborting pickup sequence.")
        return False
    
    q_goal = inverse_kinematics(tx, ty, tz)
    
    # Read actual positions instead of defaulting to [0,0,0,0]
    curr_pos_dict = read_positions(portHandler, packetHandler)
    if curr_pos_dict and all(i in curr_pos_dict for i in [11, 12, 13, 14]):
        q_start = [dynamixel_to_rad(curr_pos_dict[11]),
                   dynamixel_to_rad(curr_pos_dict[12]),
                   dynamixel_to_rad(curr_pos_dict[13]),
                   dynamixel_to_rad(curr_pos_dict[14])]
    else:
        print("Hardware missed a position read (likely motor 13). Retrying calculation...")
        return False
        
    q_above = list(q_goal)
    q_above[2] += 0.5 # Safety offset Z
    
    print("Executing Pick Sequence...")
    move_smooth(portHandler, packetHandler, q_start, q_above, 2.5)
    move_smooth(portHandler, packetHandler, q_above, q_goal, 1.5)
    set_gripper(portHandler, packetHandler, open_gripper=False) # Gripper closes picking up the object
    move_smooth(portHandler, packetHandler, q_goal, q_above, 2.5)
    
    print("Moving back to default position with payload...")
    # Default position for joints 11-14 matching set_default_positions()
    q_default = [
        dynamixel_to_rad(2048),
        dynamixel_to_rad(1227),
        dynamixel_to_rad(2524),
        dynamixel_to_rad(2414)
    ]
    move_smooth(portHandler, packetHandler, q_above, q_default, 3.0)
    
    print("Pick sequence completed.")
    return True

def main_sequence():
    pipeline, align, intrinsics = init_realsense()
    detector_params, aruco_dict, detector, OPENCV_OLD = init_aruco()
    portHandler, packetHandler = init_dynamixel()
    
    if not portHandler:
        print("Failed to connect to Dynamixel. Please verify COM port.")
        return

    # Move to default positions immediately
    set_default_positions(portHandler, packetHandler)

    camera_matrix = np.array([
        [intrinsics.fx, 0, intrinsics.ppx],
        [0, intrinsics.fy, intrinsics.ppy],
        [0, 0, 1]
    ], dtype=np.float32)
    dist_coeffs = np.zeros((4,1))
    
    success = False
    try:
        print("System ready... Displaying camera feed. Press STOP in Jupyter to exit.")
        
        # Keep searching and attempting task until it succeeds
        while not success:
            success = execute_task(pipeline, align, intrinsics, detector_params, aruco_dict, detector, OPENCV_OLD, portHandler, packetHandler, camera_matrix, dist_coeffs)
            
    except KeyboardInterrupt:
        print("\nManual interrupt received. Shutting down...")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        print("Closing port while keeping Dynamixel torque PERMANENTLY enabled.")
        portHandler.closePort()

In [9]:
main_sequence()

Moving robot to default position using software interpolation...
System ready... Displaying camera feed. Press STOP in Jupyter to exit.
Target object detected (ID: 5) at pixel (368, 252)
Base Detected! Corrected Target -> X: 0.187m, Y: 0.007m, Z: 0.589m
Executing Pick Sequence...
Moving back to default position with payload...
Pick sequence completed.
Closing port while keeping Dynamixel torque PERMANENTLY enabled.
